In [3]:
import pandas as pd
import os

In [2]:
base_dir = 'data/census'

In [14]:
dfs = []
for files in os.listdir(base_dir):
    if files.endswith('.xlsx') or files.endswith('.xls'):
        df = pd.read_excel(os.path.join(base_dir, files))
        year = int(files.split('_')[0][-4:])
        df['year'] = year
        print(year,df.shape)
        dfs.append(df)

df = pd.concat(dfs)
df = df.set_index(['year','HHNO','PPN']).reset_index()
print(df.head())
#dfs.to_csv('data/census/census_96_21.csv', index=False)

2011 (48498, 39)
2001 (43514, 36)
1996 (40423, 32)
2016 (49794, 46)
2021 (50625, 47)
2006 (45773, 36)
   year     HHNO  PPN  QRTYP  HHTYPE  TENURE  RENT  MORTHH  YRLEFT  UHSIZE  \
0  2011  4000011    0      3       1       2    99    99.0     9.0       3   
1  2011  4000011    1      3       1       2    99    99.0     9.0       3   
2  2011  4000011    2      3       1       2    99    99.0     9.0       3   
3  2011  4000011    3      3       1       2    99    99.0     9.0       3   
4  2011  4000063    0      3       1       4    99    99.0     9.0       2   

   ...  SFAREAGP1_MT  SFAREAGP2_MT  RCHI  RENG  ROTHERLANG  WCHI  WENG  \
0  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
1  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
2  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
3  ...           NaN           NaN   NaN   NaN         NaN   NaN   NaN   
4  ...           NaN           NaN   NaN   NaN         NaN 

In [20]:
ddict = pd.read_csv(os.path.join(base_dir,'data_dict.csv'))
ddict.head()

,no,field_name,description,field_width,years_available
0,1,HHNO,Household serial number,7,"2006, 2011, 2016"
1,2,PPN,Person serial number,2,"2006, 2011, 2016"
2,3,QRTYP,Type of quarters,1,"2006, 2011, 2016"
3,4,HHTYPE,Type of household,1,"2006, 2011, 2016"
4,5,TENURE,Tenure of accommodation,1,"2006, 2011, 2016"


In [24]:
completion = df.groupby('year').apply(
    lambda subset: subset.notnull().mean()
).T.merge(ddict[['field_name','description']],left_index=True,right_on='field_name',how='left')

In [26]:
completion[(completion.drop(columns=['field_name','description']) < 1).any(axis=1)]

,1996,2001,2006,2011,2016,2021,field_name,description
6.0,0.0,1.0,1.0,1.0,1.0,1.0,MORTHH,Monthly domestic household mortgage payment an...
7.0,0.0,1.0,1.0,1.0,1.0,1.0,YRLEFT,Outstanding period of mortgage payment or loan...
12.0,0.0,1.0,1.0,1.0,1.0,1.0,MORT_INC,Mortgage payment and loan repayment to income ...
21.0,0.0,1.0,1.0,1.0,1.0,1.0,ETHNIC,Ethnicity
42.0,0.0,0.0,0.0,1.0,1.0,1.0,HHCOMP11,Household composition (new classification)
NaN,0.0,0.0,0.0,1.0,1.0,0.0,INDUST_NEW,NaN
44.0,0.0,0.0,0.0,1.0,1.0,0.0,OCCUP_NEW,Occupation (ISCO-08)
45.0,1.0,1.0,1.0,1.0,0.0,1.0,INDUST,Industry (HSIC Version 1.1)
46.0,1.0,1.0,1.0,1.0,0.0,1.0,OCCUP,Occupation (ISCO-88)
33.0,0.0,0.0,0.0,0.0,1.0,1.0,SFAREAGP1_MT,Floor area of accommodation (in square metres)


In [30]:
completion[(completion.drop(columns=['field_name','description']) == 1).all(axis=1)]

,1996,2001,2006,2011,2016,2021,field_name,description
0.0,1.0,1.0,1.0,1.0,1.0,1.0,HHNO,Household serial number
1.0,1.0,1.0,1.0,1.0,1.0,1.0,PPN,Person serial number
2.0,1.0,1.0,1.0,1.0,1.0,1.0,QRTYP,Type of quarters
3.0,1.0,1.0,1.0,1.0,1.0,1.0,HHTYPE,Type of household
4.0,1.0,1.0,1.0,1.0,1.0,1.0,TENURE,Tenure of accommodation
5.0,1.0,1.0,1.0,1.0,1.0,1.0,RENT,Monthly domestic household rent
8.0,1.0,1.0,1.0,1.0,1.0,1.0,UHSIZE,Household size
9.0,1.0,1.0,1.0,1.0,1.0,1.0,DJHHINC,Monthly domestic household income
10.0,1.0,1.0,1.0,1.0,1.0,1.0,DJHHCOMP,Household composition (old classification)
11.0,1.0,1.0,1.0,1.0,1.0,1.0,RENT_INC,Rent to income ratio


In [32]:
df.to_csv(os.path.join(base_dir,'census_96_21.csv'),index=False)

In [45]:
import requests
import json



url = "https://www.censtatd.gov.hk/api/post.php"

dfs = []
ids = ['140-09001','140-09002','140-09003','140-09004','140-09005']
years = ['1999','2004','2009','2014','2019']
for id,year in zip(ids,years):
  parameters ={
    "cv": {
      "GROUP": [
        "S1",
        "S2",
        "S3",
        "S4",
        "S5",
        "S6",
        "S7",
        "S8",
        "S9"
      ],
      "LQTYPE": [
        "1",
        "2",
        "3"
      ]
    },
    "sv": {
      "HOUSE_EXPD": [
        "Raw_hkd_d"
      ]
    },
    "period": {
      "start": year
    },
    "id": id,
    "lang": "en"
  }
  data = {'query': json.dumps(parameters)}
  r = requests.post(url, data=data, timeout=20)
  print(year,r.status_code)
  try:
    data = r.json()['dataSet']
    df = pd.DataFrame(data)
    print(df.describe())
    dfs.append(df)
  except:
    print(r.text)
    break




1999 200
             figure
count     40.000000
mean    4202.250000
std     6352.400522
min      164.000000
25%      724.250000
50%     1251.000000
75%     5186.000000
max    27239.000000
2004 200
             figure
count     40.000000
mean    3537.150000
std     5364.973886
min      138.000000
25%      683.000000
50%     1134.000000
75%     4242.000000
max    23920.000000
2009 200
            figure
count     40.00000
mean    4009.75000
std     6284.33141
min      117.00000
25%      644.00000
50%     1154.00000
75%     5052.75000
max    28715.00000
2014 200
             figure
count     40.000000
mean    5185.275000
std     8218.943809
min      131.000000
25%      750.500000
50%     1270.000000
75%     6507.750000
max    36728.000000
2019 200
             figure
count     40.000000
mean    5538.675000
std     8775.684534
min      126.000000
25%      746.500000
50%     1448.500000
75%     7265.250000
max    37895.000000


In [52]:
big_df = pd.concat(dfs)
big_df = big_df[big_df['GROUPDesc']!='Total']
big_df = big_df[big_df['LQTYPEDesc']!='Total']

In [53]:
big_df.to_csv('data/household/household_expenditure_99_19_grouped.csv',index=False)